In [9]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import numpy as np
from pathlib import Path
import json
import pandas as pd
from tfmbench.datasets import BaseTabularDataset as TabularDataset
from tfmbench.evaluate import evaluate
from tfmbench.benchmark import benchmark_models
from tfmbench.datasets.talent import load_talent_dataset
from tfmbench.utils import get_tab_split

In [18]:
from pathlib import Path
import ast
import struct
import json

import numpy as np
import pandas as pd


def get_npy_info(path):
    """
    Read shape and dtype from a .npy file without loading it.
    """
    path = Path(path)

    with open(path, "rb") as f:

        magic = f.read(6)

        if magic != b"\x93NUMPY":
            raise ValueError(f"{path} is not a valid .npy file")

        major = int.from_bytes(f.read(1), "little")
        minor = int.from_bytes(f.read(1), "little")

        if major == 1:
            header_len = struct.unpack("<H", f.read(2))[0]
            encoding = "latin1"

        elif major in (2, 3):
            header_len = struct.unpack("<I", f.read(4))[0]
            encoding = "utf-8" if major == 3 else "latin1"

        else:
            raise ValueError(
                f"Unsupported .npy version {major}.{minor}"
            )

        header = f.read(header_len).decode(encoding)

        header_dict = ast.literal_eval(header)

        shape = tuple(header_dict["shape"])
        dtype = np.dtype(header_dict["descr"])

    return shape, dtype


def get_talent_dataset_stats(root):

    root = Path(root)
    rows = []

    for dataset_dir in sorted(root.iterdir()):

        if not dataset_dir.is_dir():
            continue

        if not (dataset_dir / "y_train.npy").exists():
            continue

        try:
            
            info_path = dataset_dir / "info.json"

            if info_path.exists():
                with open(info_path, "r") as f:
                    info = json.load(f)
            else:
                info = {}

            task_type = info.get("task_type", "unknown")

            split_sizes = {}

            for split in ["train", "val", "test"]:

                y_path = dataset_dir / f"y_{split}.npy"

                if y_path.exists():

                    shape, _ = get_npy_info(y_path)

                    split_sizes[split] = shape[0]

                else:

                    split_sizes[split] = 0


            n_train = split_sizes["train"]
            n_val = split_sizes["val"]
            n_test = split_sizes["test"]

            total_rows = (
                n_train
                + n_val
                + n_test
            )

            n_num_features = 0
            n_cat_features = 0

            n_path = dataset_dir / "N_train.npy"
            c_path = dataset_dir / "C_train.npy"

            if n_path.exists():

                shape, _ = get_npy_info(n_path)

                if len(shape) > 1:
                    n_num_features = shape[1]

            if c_path.exists():

                shape, _ = get_npy_info(c_path)

                if len(shape) > 1:
                    n_cat_features = shape[1]

            n_features = (
                n_num_features
                + n_cat_features
            )

            y_train = np.load(
                dataset_dir / "y_train.npy",
                allow_pickle=True,
            )

            y_train = np.asarray(
                y_train
            ).reshape(-1)

            is_regression = (
                "regression"
                in str(task_type).lower()
            )

            if is_regression:

                n_classes = None
                class_counts = None

                y_numeric = y_train.astype(float)

                target_mean = float(
                    np.mean(y_numeric)
                )

                target_std = float(
                    np.std(y_numeric)
                )

            else:

                unique_values, counts = np.unique(
                    y_train,
                    return_counts=True,
                )

                n_classes = len(unique_values)

                class_counts = {
                    str(cls): int(count)
                    for cls, count in zip(
                        unique_values,
                        counts,
                    )
                }

                target_mean = None
                target_std = None

            train_fraction = (
                n_train / total_rows
                if total_rows
                else None
            )

            rows_per_feature = (
                total_rows / n_features
                if n_features
                else None
            )


            rows.append({
                "dataset": dataset_dir.name,
                "task": task_type,

                "total_rows": total_rows,
                "train_rows": n_train,
                "val_rows": n_val,
                "test_rows": n_test,

                "n_features": n_features,
                "n_numeric_features": n_num_features,
                "n_categorical_features": n_cat_features,

                "n_classes": n_classes,
                #"class_counts_train": class_counts,
                #"target_mean": target_mean,
                #"target_std": target_std,
            })

        except Exception as e:

            print(
                f"Failed to process {dataset_dir.name}: "
                f"{type(e).__name__}: {e}"
            )

    df = pd.DataFrame(rows)

    if not df.empty:

        df = (
            df
            .sort_values(
                "total_rows",
                ascending=False,
            )
            .reset_index(drop=True)
        )

    return df

In [17]:
stats_df = get_talent_dataset_stats(
    "./data/datasets-talent-large"
)

stats_df

,dataset,task,total_rows,train_rows,val_rows,test_rows,n_features,n_numeric_features,n_categorical_features,n_classes,class_counts_train,train_fraction,rows_per_feature,target_mean,target_std
0,Airlines_DepDelay_10M,regression,10000000,6400000,1600000,2000000,9,6,3,NaN,None,0.640000,1.111111e+06,8.219144e+00,29.466710
1,KDDCup99,multiclass,4898431,3134995,783749,979687,41,32,9,23.0,"{'0': 1409, '1': 19, '2': 5, '3': 34, '4': 8, ...",0.640000,1.194739e+05,NaN,NaN
2,sf-police-incidents,binclass,2215023,1417614,354404,443005,8,3,5,2.0,"{'0': 1245250, '1': 172364}",0.640000,2.768779e+05,NaN,NaN
3,microsoft,regression,1200192,723412,235259,241521,136,136,0,NaN,None,0.602747,8.824941e+03,6.653069e-01,0.822127
4,poker-hand,multiclass,1025009,656005,164002,205002,10,10,0,10.0,"{'0': 328768, '1': 5, '2': 277182, '3': 31250,...",0.639999,1.025009e+05,NaN,NaN
5,Higgs,binclass,1000000,640000,160000,200000,28,28,0,2.0,"{'0': 300851, '1': 339149}",0.640000,3.571429e+04,NaN,NaN
6,BNG(credit-a),binclass,1000000,640000,160000,200000,15,6,9,2.0,"{'0': 285435, '1': 354565}",0.640000,6.666667e+04,NaN,NaN
7,Smoking_and_Drinking_Dataset_with_body_signal,binclass,991346,634460,158616,198270,23,22,1,2.0,"{'0': 317348, '1': 317112}",0.639999,4.310200e+04,NaN,NaN
8,yahoo,regression,709877,473134,71083,165660,699,699,0,NaN,None,0.666501,1.015561e+03,1.233974e+00,0.983204
9,Data_Science_for_Good_Kiva_Crowdfunding,multiclass,671205,429571,107393,134241,11,7,4,4.0,"{'0': 45265, '1': 164581, '2': 219339, '3': 386}",0.640000,6.101864e+04,NaN,NaN


In [19]:
stats_df = get_talent_dataset_stats(
    "./data/datasets-talent-highdim-cv"
)

stats_df

,dataset,task,total_rows,train_rows,val_rows,test_rows,n_features,n_numeric_features,n_categorical_features,n_classes
0,gisette_1,binclass,7000,4480,1120,1400,5000,5000,0,2
1,gisette_5,binclass,7000,4480,1120,1400,5000,5000,0,2
2,gisette_4,binclass,7000,4480,1120,1400,5000,5000,0,2
3,gisette_3,binclass,7000,4480,1120,1400,5000,5000,0,2
4,gisette_2,binclass,7000,4480,1120,1400,5000,5000,0,2
...,...,...,...,...,...,...,...,...,...,...
125,GLIOMA_5,multiclass,50,32,8,10,4434,4434,0,4
126,GLIOMA_4,multiclass,50,32,8,10,4434,4434,0,4
127,GLIOMA_3,multiclass,50,32,8,10,4434,4434,0,4
128,GLIOMA_2,multiclass,50,32,8,10,4434,4434,0,4
